In [1]:
# Fit missing-value handling, scaling, and category encoding on training data only.
# Save the frozen preprocessing pipeline and train, validation, and test matrices.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path:
    _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash
from project_config import ID_COLUMNS
from pathlib import Path
import hashlib
import json
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/modeling_dataset_unprocessed.csv").exists():
    ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
MODELS = project_paths(ROOT).models
KEYS = ["company_id", "prediction_date"]
SPLITS = ["train", "validation", "test"]
NUMERIC_PREFIXES = ("original_", "derived_", "trend_", "market_", "macro_")
CATEGORICAL_COLUMNS = ["sector", "industry"]
MISSING_CATEGORY = "__MISSING__"
input_hashes = {}


def read_input(path):
    input_hashes[path] = file_hash(path)
    return pd.read_csv(path, dtype={"company_id": "string", "metadata_event_id": "string"})

model = read_input(PROCESSED / "modeling_dataset_unprocessed.csv")
if model.empty or model[KEYS].isna().any().any() or model.duplicated(KEYS).any() or not model.metadata_event_id.is_unique:
    raise ValueError("Require uniquely identified company-events")
model_before = model.copy(deep=True)
ids = {name: read_input(PROCESSED / f"{name}_ids.csv") for name in SPLITS}
model_indexed = model.set_index(KEYS)
cohorts = {}
for name in SPLITS:
    frame = ids[name]
    if not set(ID_COLUMNS).issubset(frame.columns) or frame.empty or frame[ID_COLUMNS].isna().any().any():
        raise ValueError(f"{name}: missing split identifiers/metadata")
    if frame.duplicated(KEYS).any() or not frame.metadata_event_id.is_unique or not frame.split.eq(name).all():
        raise ValueError(f"{name}: invalid unique membership")
    index = pd.MultiIndex.from_frame(frame[KEYS])
    if len(index.difference(model_indexed.index)):
        raise ValueError(f"{name}: split contains a company-event absent from the dataset")
    # Keep ID-file order without inserting columns into a fragmented wide DataFrame.
    values = model_indexed.reindex(index).copy()
    values.index = frame.index
    cohort = pd.concat([frame[KEYS], values], axis=1)
    if not cohort.target.isin([0, 1]).all(): raise ValueError(f"{name}: nonbinary target")
    cohorts[name] = cohort
for i, left in enumerate(SPLITS):
    for right in SPLITS[i+1:]:
        for column in ["metadata_event_id", "company_id", "company_group_id"]:
            if not set(ids[left][column]).isdisjoint(ids[right][column]):
                raise ValueError(f"{column} leaks between {left} and {right}")
for left, right in [("train", "validation"), ("validation", "test")]:
    if not pd.to_datetime(ids[left].prediction_date).max() < pd.to_datetime(ids[right].prediction_date).min():
        raise ValueError("Prediction chronology violated")
    if not pd.to_datetime(ids[left].metadata_outcome_end_date).max() < pd.to_datetime(ids[right].prediction_date).min():
        raise ValueError("Earlier split's outcome labels are not available before the next split")

numeric_candidates = [column for column in model if column.startswith(NUMERIC_PREFIXES)]
if not numeric_candidates or not set(CATEGORICAL_COLUMNS).issubset(model.columns):
    raise ValueError("Missing expected numerical/categorical predictors")
if not all(pd.api.types.is_numeric_dtype(model[column]) for column in numeric_candidates):
    raise ValueError("Numerical predictors must already have numerical types")
for name, cohort in cohorts.items():
    if np.isinf(cohort[numeric_candidates].to_numpy(dtype=float)).any():
        raise ValueError(f"{name}: infinite source predictors require upstream correction")
    for column in [c for c in cohort if c.startswith("missing_")]:
        original_column = "original_" + column.removeprefix("missing_")
        if original_column not in cohort or not cohort[column].eq(cohort[original_column].isna().astype(int)).all():
            raise ValueError(f"{name}: inconsistent original missingness flag {column}")

# This is the only data used to select columns/vocabularies or fit parameters.
X = {name: cohort[numeric_candidates + CATEGORICAL_COLUMNS].copy() for name, cohort in cohorts.items()}
training_observed = X["train"][numeric_candidates].notna().any()
numeric_columns = training_observed.index[training_observed].tolist()
dropped_numeric_columns = training_observed.index[~training_observed].tolist()
if not numeric_columns: raise ValueError("No observed training numerical predictors")
category_vocabularies = {}
for column in CATEGORICAL_COLUMNS:
    observed = X["train"][column].dropna()
    if not observed.map(lambda value: isinstance(value, str)).all():
        raise ValueError(f"{column}: expected string categories")
    if observed.eq(MISSING_CATEGORY).any():
        raise ValueError(f"Reserved category collides with observed {column}")
    category_vocabularies[column] = sorted(observed.unique().tolist()) + [MISSING_CATEGORY]


In [3]:
# Parallel branches retain scaled values, unscaled missingness, and categorical dummies.
# Median statistics, scaling statistics, and every learned vocabulary use train only.
numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
    ("standardize", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value=MISSING_CATEGORY, keep_empty_features=True)),
    ("one_hot", OneHotEncoder(categories=[category_vocabularies[c] for c in CATEGORICAL_COLUMNS],
                              handle_unknown="ignore", sparse_output=False, dtype=np.float64)),
])
columns = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("missing", MissingIndicator(features="all", sparse=False, error_on_new=False), numeric_columns),
    ("categorical", categorical_pipeline, CATEGORICAL_COLUMNS),
], remainder="drop", sparse_threshold=0, verbose_feature_names_out=True)
preprocessing_pipeline = Pipeline([("preprocess", columns)])
preprocessing_pipeline.fit(X["train"])
feature_names = preprocessing_pipeline.get_feature_names_out().tolist()
if len(feature_names) != len(set(feature_names)): raise ValueError("Duplicate transformed predictor names")

# Record metadata using serializable built-in values, not notebook-specific objects.
preprocessing_pipeline.project_schema_ = {
    "input_feature_columns": numeric_candidates + CATEGORICAL_COLUMNS,
    "retained_numeric_columns": numeric_columns,
    "dropped_training_empty_numeric_columns": dropped_numeric_columns,
    "categorical_columns": CATEGORICAL_COLUMNS,
    "category_vocabularies": category_vocabularies,
    "missing_category": MISSING_CATEGORY,
    "matrix_feature_columns": feature_names,
    "matrix_metadata_columns": ID_COLUMNS,
    "fit_split": "train",
    "training_event_ids": ids["train"].metadata_event_id.tolist(),
    "source_sha256": input_hashes[PROCESSED / "modeling_dataset_unprocessed.csv"],
    "split_file_sha256": {name: input_hashes[PROCESSED / f"{name}_ids.csv"] for name in SPLITS},
    "split_counts": {name: {"events": len(ids[name]), "bankrupt": int(ids[name].target.sum())} for name in SPLITS},
    "versions": {"python": platform.python_version(), "scikit_learn": sklearn.__version__, "joblib": joblib.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "unseen_category_policy": "all-zero one-hot block (not missing)",
    "missingness_policy": "one binary indicator per retained numeric field before imputation",
    "matrix_role": "Logistic-regression-ready predictors plus separate ID/target metadata; no classifier fitted",
}
# Compare the state hash before/after transforms; held-out data cannot refit steps.
state_after_fit = joblib.hash(preprocessing_pipeline)
transformed = {name: preprocessing_pipeline.transform(X[name]) for name in SPLITS}


In [4]:
# Verify each fitted parameter directly against the training cohort.
fitted_columns = preprocessing_pipeline.named_steps["preprocess"]
fitted_imputer = fitted_columns.named_transformers_["numeric"].named_steps["impute"]
fitted_scaler = fitted_columns.named_transformers_["numeric"].named_steps["standardize"]
fitted_indicator = fitted_columns.named_transformers_["missing"]
fitted_encoder = fitted_columns.named_transformers_["categorical"].named_steps["one_hot"]
expected_medians = X["train"][numeric_columns].median(axis=0).to_numpy(dtype=float)
expected_imputed = X["train"][numeric_columns].fillna(pd.Series(expected_medians, index=numeric_columns)).to_numpy(dtype=float)

numeric_slice = fitted_columns.output_indices_["numeric"]
indicator_slice = fitted_columns.output_indices_["missing"]
categorical_slice = fitted_columns.output_indices_["categorical"]
for name in SPLITS:
    values = np.asarray(transformed[name], dtype=float)
    indicators = values[:, indicator_slice]
    expected_scaled = (X[name][numeric_columns].fillna(pd.Series(expected_medians, index=numeric_columns)).to_numpy(dtype=float) - fitted_scaler.mean_) / fitted_scaler.scale_
constant = fitted_scaler.var_ == 0

# Preserve the training-fitted state after transforming held-out rows.

# Report unknown held-out categories without modifying the fitted vocabulary.
category_coverage = []
for name in SPLITS:
    for column in CATEGORICAL_COLUMNS:
        values = X[name][column]
        unknown = values.notna() & ~values.isin(category_vocabularies[column])
        category_coverage.append({"split": name, "categorical_field": column,
                                  "missing_rows": int(values.isna().sum()), "unseen_category_rows": int(unknown.sum()),
                                  "unseen_values": sorted(values.loc[unknown].unique().tolist())})


In [5]:
# Assemble identical-width matrices with membership/labels kept separate from predictors.
matrices = {}
for name in SPLITS:
    predictor_matrix = pd.DataFrame(transformed[name], columns=feature_names, index=ids[name].index)
    matrices[name] = pd.concat([ids[name][ID_COLUMNS].copy(), predictor_matrix], axis=1)

# Save a self-contained built-in sklearn Pipeline, then verify a fresh loaded instance.
MODELS.mkdir(parents=True, exist_ok=True)
artifact_path = MODELS / "preprocessing_pipeline.joblib"
joblib.dump(preprocessing_pipeline, artifact_path, compress=3)
loaded_pipeline = joblib.load(artifact_path)
for name, frame in matrices.items():
    frame.to_csv(PROCESSED / f"{name}_matrix.csv", index=False)
